# Stochastic Optimization — Theory Notebook

> *"The best thing about SGD is that it works. The worst thing is that nobody fully understands why."*

Interactive demonstrations of SGD convergence, mini-batch variance, SVRG, SAGA, distributed SGD, and the generalization gap. Companion to [notes.md](notes.md).

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.optimize as opt

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

if HAS_MPL:
    mpl.rcParams.update({
        'figure.figsize': (10, 6),
        'figure.dpi': 120,
        'font.size': 13,
        'axes.titlesize': 15,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
    })

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)
print('Setup complete.')

---

## 1. SGD vs. Full-Batch GD

Compare stochastic gradient descent with full-batch gradient descent on a quadratic problem.

In [ ]:
np.random.seed(42)
n, d = 10000, 50
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)
def grad_full(x): return A.T @ (A @ x - b) / n

def sgd_step(x, lr, batch_size=1):
    idx = np.random.randint(0, n, size=batch_size)
    g = A[idx].T @ (A[idx] @ x - b[idx]) / batch_size
    return x - lr * g

# Full-batch GD
x_gd = np.zeros(d)
L = np.linalg.norm(A.T @ A) / n
eta_gd = 1.0 / L
err_gd = []
for t in range(200):
    x_gd = x_gd - eta_gd * grad_full(x_gd)
    err_gd.append(f_full(x_gd))

# SGD
x_sgd = np.zeros(d)
eta_sgd = 0.001
err_sgd = []
for t in range(200 * n):
    x_sgd = sgd_step(x_sgd, eta_sgd, batch_size=1)
    if (t + 1) % n == 0:
        err_sgd.append(f_full(x_sgd))

err_gd = np.array(err_gd)
err_sgd = np.array(err_sgd)

print(f'Full-batch GD after 200 iterations: {err_gd[-1]:.2e}')
print(f'SGD after 200 epochs: {err_sgd[-1]:.2e}')

ok = err_sgd[-1] < err_gd[-1] * 10
print(f"\n{'PASS' if ok else 'FAIL'} — SGD makes progress")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    gd_epochs = np.arange(1, len(err_gd) + 1)
    sgd_epochs = np.arange(1, len(err_sgd) + 1)
    
    ax.plot(gd_epochs, err_gd, color=COLORS['neutral'], label='Full-batch GD')
    ax.plot(sgd_epochs, err_sgd, color=COLORS['primary'], label='SGD (batch=1)')
    
    ax.set_yscale('log')
    ax.set_title('SGD vs Full-batch GD (same gradient evaluations)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Objective value')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 2. Mini-Batch Variance

Demonstrate how batch size affects gradient variance.

In [ ]:
np.random.seed(42)
n, d = 10000, 50
A = np.random.randn(n, d)
x = np.random.randn(d)
b = A @ x + 0.1 * np.random.randn(n)

full_grad = A.T @ (A @ x - b) / n

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
variances = []

for B in batch_sizes:
    grads = []
    for _ in range(500):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        grads.append(g)
    grads = np.array(grads)
    var = np.mean(np.sum((grads - full_grad)**2, axis=1))
    variances.append(var)

variances = np.array(variances)

print('Batch size variance analysis:')
print(f"{'B':>6} | {'Variance':>12} | {'Ratio to B=1':>14}")
print('-' * 40)
for B, v in zip(batch_sizes, variances):
    ratio = variances[0] / v if v > 0 else float('inf')
    print(f'{B:>6} | {v:>12.6f} | {ratio:>14.1f}x')

ok = variances[-1] < variances[0] / 100
print(f"\n{'PASS' if ok else 'FAIL'} — variance decreases with batch size")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(batch_sizes, variances, 'o-', color=COLORS['primary'])
    ax.plot(batch_sizes, variances[0] / np.array(batch_sizes), '--',
            color=COLORS['neutral'], label='σ²/B (theoretical)')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('Gradient variance vs batch size')
    ax.set_xlabel('Batch size B')
    ax.set_ylabel('Variance')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 3. SGD Convergence Rate

Verify the $O(1/T)$ convergence rate for strongly convex functions.

In [ ]:
np.random.seed(42)
d = 20
mu = 0.1
L = 1.0
sigma = 0.5

D = np.linspace(mu, L, d)
x_star = np.random.randn(d)

def f_sc(x): return 0.5 * np.sum(D * (x - x_star)**2)
def grad_sc(x): return D * (x - x_star)

def stochastic_grad(x, sigma):
    return grad_sc(x) + sigma * np.random.randn(d)

T = 10000
x = np.zeros(d)
err_sgd = []

for t in range(T):
    eta_t = 2.0 / (mu * (t + 2))
    g = stochastic_grad(x, sigma)
    x = x - eta_t * g
    err_sgd.append(np.linalg.norm(x - x_star)**2)

err_sgd = np.array(err_sgd)

print(f'SGD on strongly convex quadratic:')
print(f'Initial error: {err_sgd[0]:.4f}')
print(f'Final error: {err_sgd[-1]:.6f}')
print(f'Theoretical bound: O(1/T) = {1/T:.6f}')

T_vals = np.array([100, 500, 1000, 5000, 10000])
print(f"\n{'T':>6} | {'Error':>12} | {'T * Error':>12}")
print('-' * 35)
for T_val in T_vals:
    print(f'{T_val:>6} | {err_sgd[T_val-1]:>12.6f} | {T_val*err_sgd[T_val-1]:>12.6f}')

ok = err_sgd[-1] < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — SGD converges")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_sgd, color=COLORS['primary'])
    ax.plot(steps, err_sgd[0] / steps * steps[0], '--',
            color=COLORS['neutral'], label='O(1/T) rate')
    
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_title('SGD convergence: O(1/T) for strongly convex')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('||x - x*||²')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 4. SGD with Momentum

Analyze how momentum affects SGD convergence in the presence of noise.

In [ ]:
np.random.seed(42)
d = 20
mu, L, sigma = 0.1, 1.0, 0.5
D = np.linspace(mu, L, d)
x_star = np.random.randn(d)

def grad_sc(x): return D * (x - x_star)
def stochastic_grad(x, sigma):
    return grad_sc(x) + sigma * np.random.randn(d)

T = 5000

# Vanilla SGD
x_sgd = np.zeros(d)
err_sgd = []
for t in range(T):
    eta = 0.01
    x_sgd = x_sgd - eta * stochastic_grad(x_sgd, sigma)
    err_sgd.append(np.linalg.norm(x_sgd - x_star)**2)

# SGD + Momentum
x_mom = np.zeros(d)
v = np.zeros(d)
beta = 0.9
err_mom = []
for t in range(T):
    eta = 0.01
    g = stochastic_grad(x_mom, sigma)
    v = beta * v + g
    x_mom = x_mom - eta * v
    err_mom.append(np.linalg.norm(x_mom - x_star)**2)

err_sgd = np.array(err_sgd)
err_mom = np.array(err_mom)

print(f'After {T} iterations:')
print(f'  SGD:        error = {err_sgd[-1]:.6f}')
print(f'  SGD+Momentum: error = {err_mom[-1]:.6f}')

ok = err_mom[-1] < err_sgd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — momentum helps")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    window = 100
    err_sgd_smooth = np.convolve(err_sgd, np.ones(window)/window, mode='valid')
    err_mom_smooth = np.convolve(err_mom, np.ones(window)/window, mode='valid')
    steps_smooth = np.arange(window, T + 1)
    
    ax.plot(steps_smooth, err_sgd_smooth, color=COLORS['neutral'], label='SGD')
    ax.plot(steps_smooth, err_mom_smooth, color=COLORS['primary'], label='SGD + Momentum')
    
    ax.set_yscale('log')
    ax.set_title('SGD vs SGD+Momentum (smoothed)')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('||x - x*||²')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 5. SVRG: Stochastic Variance Reduced Gradient

Implement and analyze SVRG for logistic regression.

In [ ]:
np.random.seed(42)
n, d = 500, 20
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = (X @ w_true > 0).astype(float) * 2 - 1

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def f_i(w, i):
    z = y[i] * X[i] @ w
    return np.log(1 + np.exp(-np.clip(z, -500, 500)))

def grad_f_i(w, i):
    z = y[i] * X[i] @ w
    s = sigmoid(-z)
    return -y[i] * s * X[i]

def full_grad(w):
    return np.mean([grad_f_i(w, i) for i in range(n)], axis=0)

def full_loss(w):
    return np.mean([f_i(w, i) for i in range(n)])

# SVRG
w = np.zeros(d)
m = 2 * n
eta = 0.01
n_epochs = 20
loss_hist = []

for s in range(n_epochs):
    full_g = full_grad(w)
    w_snapshot = w.copy()
    for t in range(m):
        i = np.random.randint(n)
        g_current = grad_f_i(w, i)
        g_snapshot = grad_f_i(w_snapshot, i)
        g_svrg = g_current - g_snapshot + full_g
        w = w - eta * g_svrg
    loss_hist.append(full_loss(w))

loss_hist = np.array(loss_hist)

print(f'SVRG for logistic regression:')
print(f'Initial loss: {loss_hist[0]:.6f}')
print(f'Final loss: {loss_hist[-1]:.6f}')
print(f'Convergence rate (per epoch):')
for s in range(1, min(10, len(loss_hist))):
    ratio = loss_hist[s] / loss_hist[s-1] if loss_hist[s-1] > 0 else 0
    print(f'  Epoch {s}: loss={loss_hist[s]:.6f}, ratio={ratio:.4f}')

ok = loss_hist[-1] < loss_hist[0] / 10
print(f"\n{'PASS' if ok else 'FAIL'} — SVRG converges")

---

## 6. SAGA: Stochastic Average Gradient Amélioré

Compare SAGA with SVRG and vanilla SGD.

In [ ]:
np.random.seed(42)

def saga_solver(n, d, grad_f_i, full_loss, max_iter=10000, eta=0.01):
    w = np.zeros(d)
    grad_table = np.array([grad_f_i(w, i) for i in range(n)])
    avg_grad = np.mean(grad_table, axis=0)
    loss_hist = []
    for t in range(max_iter):
        i = np.random.randint(n)
        g_current = grad_f_i(w, i)
        g_saga = g_current - grad_table[i] + avg_grad
        w = w - eta * g_saga
        grad_table[i] = g_current
        avg_grad = avg_grad + (g_current - grad_table[i]) / n
        if (t + 1) % (2 * n) == 0:
            loss_hist.append(full_loss(w))
    return np.array(loss_hist)

def sgd_solver(n, d, grad_f_i, full_loss, max_iter=10000, eta=0.01):
    w = np.zeros(d)
    loss_hist = []
    for t in range(max_iter):
        i = np.random.randint(n)
        g = grad_f_i(w, i)
        w = w - eta * g
        if (t + 1) % (2 * n) == 0:
            loss_hist.append(full_loss(w))
    return np.array(loss_hist)

loss_saga = saga_solver(n, d, grad_f_i, full_loss, max_iter=20000, eta=0.01)
loss_sgd = sgd_solver(n, d, grad_f_i, full_loss, max_iter=20000, eta=0.01)

print(f'Comparison (logistic regression, n={n}, d={d}):')
print(f'  SAGA final loss: {loss_saga[-1]:.6f}')
print(f'  SGD final loss:  {loss_sgd[-1]:.6f}')

ok = loss_saga[-1] < loss_sgd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — SAGA outperforms SGD")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = np.arange(1, len(loss_saga) + 1)
    
    ax.plot(epochs, loss_saga, color=COLORS['primary'], label='SAGA')
    ax.plot(epochs, loss_sgd, color=COLORS['neutral'], label='SGD')
    
    ax.set_yscale('log')
    ax.set_title('SAGA vs SGD convergence')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 7. Linear Scaling Rule

Verify that scaling the learning rate with batch size maintains similar training dynamics.

In [ ]:
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

base_lr = 0.01
base_batch = 32
batch_sizes = [32, 64, 128, 256]
results = {}

for B in batch_sizes:
    lr = base_lr * B / base_batch
    x = np.zeros(d)
    err = []
    total_steps = 10000
    
    for t in range(total_steps):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        x = x - lr * g
        if (t + 1) % 100 == 0:
            err.append(f_full(x))
    results[B] = np.array(err)

print(f'Linear scaling rule verification:')
print(f"{'Batch':>6} | {'LR':>8} | {'Final loss':>12}")
print('-' * 35)
for B in batch_sizes:
    lr = base_lr * B / base_batch
    print(f'{B:>6} | {lr:>8.4f} | {results[B][-1]:>12.6f}')

final_losses = [results[B][-1] for B in batch_sizes]
ok = max(final_losses) / min(final_losses) < 2.0
print(f"\n{'PASS' if ok else 'FAIL'} — losses are similar across batch sizes")

---

## 8. Generalization Gap: Small vs Large Batch

Demonstrate the generalization gap between small-batch and large-batch SGD.

In [ ]:
np.random.seed(42)
n_train, n_test, d = 500, 500, 50

X_train = np.random.randn(n_train, d)
X_test = np.random.randn(n_test, d)
w_true = np.random.randn(d)
y_train = np.sign(X_train @ w_true + 0.5 * np.random.randn(n_train))
y_test = np.sign(X_test @ w_true + 0.5 * np.random.randn(n_test))

def logistic_loss(w, X, y):
    z = y * (X @ w)
    return np.mean(np.log(1 + np.exp(-np.clip(z, -500, 500))))

batch_sizes = [1, 4, 16, 64, 256]
train_losses = []
test_losses = []

for B in batch_sizes:
    w = np.zeros(d)
    lr = 0.01 * B
    epochs = 100
    
    for epoch in range(epochs):
        for _ in range(n_train // B):
            idx = np.random.randint(0, n_train, size=B)
            z = y_train[idx] * (X_train[idx] @ w)
            s = 1.0 / (1.0 + np.exp(np.clip(z, -500, 500)))
            g = -(y_train[idx] * s) @ X_train[idx] / B
            w = w - lr * g
    
    train_losses.append(logistic_loss(w, X_train, y_train))
    test_losses.append(logistic_loss(w, X_test, y_test))

print(f'Generalization gap experiment:')
print(f"{'Batch':>6} | {'Train loss':>12} | {'Test loss':>12} | {'Gap':>8}")
print('-' * 50)
for B, tl, tel in zip(batch_sizes, train_losses, test_losses):
    gap = tel - tl
    print(f'{B:>6} | {tl:>12.6f} | {tel:>12.6f} | {gap:>8.6f}')

ok = test_losses[0] < test_losses[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — small batch generalizes better")

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    x_pos = np.arange(len(batch_sizes))
    width = 0.35
    
    bars1 = ax.bar(x_pos - width/2, train_losses, width, label='Train loss',
                   color=COLORS['primary'], alpha=0.7)
    bars2 = ax.bar(x_pos + width/2, test_losses, width, label='Test loss',
                   color=COLORS['secondary'], alpha=0.7)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(batch_sizes)
    ax.set_xlabel('Batch size')
    ax.set_ylabel('Logistic loss')
    ax.set_title('Generalization gap: small vs large batch')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 9. Distributed SGD Simulation

Simulate distributed SGD with gradient averaging across workers.

In [ ]:
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

K_values = [1, 2, 4, 8, 16]
results_dist = {}

for K in K_values:
    data_partitions = np.array_split(np.arange(n), K)
    x_workers = [np.zeros(d) for _ in range(K)]
    lr = 0.001 * K
    
    for step in range(5000):
        grads = []
        for k in range(K):
            idx = np.random.choice(data_partitions[k], size=32)
            g = A[idx].T @ (A[idx] @ x_workers[k] - b[idx]) / 32
            grads.append(g)
        
        g_avg = np.mean(grads, axis=0)
        for k in range(K):
            x_workers[k] = x_workers[k] - lr * g_avg
        
        if (step + 1) % 500 == 0:
            loss = f_full(np.mean(x_workers, axis=0))
            if K not in results_dist:
                results_dist[K] = []
            results_dist[K].append(loss)

print(f'Distributed SGD simulation:')
print(f"{'Workers':>7} | {'Final loss':>12}")
print('-' * 25)
for K in K_values:
    print(f'{K:>7} | {results_dist[K][-1]:>12.6f}')

ok = all(results_dist[K][-1] < 0.1 for K in K_values)
print(f"\n{'PASS' if ok else 'FAIL'} — all configurations converge")

---

## 10. Critical Batch Size

Identify the critical batch size where diminishing returns begin.

In [ ]:
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
speedups = []

for B in batch_sizes:
    x = np.zeros(d)
    lr = 0.001 * B
    steps_per_epoch = n // B
    total_steps = 10 * steps_per_epoch
    for t in range(total_steps):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        x = x - lr * g
    loss = f_full(x)
    speedups.append(loss)

speedups = np.array(speedups)

print(f'Critical batch size analysis:')
print(f"{'B':>6} | {'Loss after 10 epochs':>22}")
print('-' * 35)
for B, loss in zip(batch_sizes, speedups):
    print(f'{B:>6} | {loss:>22.6f}')

for i in range(1, len(speedups)):
    improvement = (speedups[i-1] - speedups[i]) / speedups[i-1]
    if improvement < 0.1:
        print(f'\nCritical batch size ≈ {batch_sizes[i]}')
        break

ok = speedups[-1] < speedups[0]
print(f"\n{'PASS' if ok else 'FAIL'} — larger batches improve convergence")

---

## Summary

This notebook demonstrated:

1. **SGD vs Full-batch GD** — SGD makes more progress per unit of compute
2. **Mini-batch variance** — Variance decreases as $\sigma^2/B$
3. **SGD convergence** — $O(1/T)$ rate for strongly convex functions
4. **SGD with momentum** — Momentum reduces variance and accelerates convergence
5. **SVRG** — Variance reduction with periodic full gradient computation
6. **SAGA** — Variance reduction without full gradient computation
7. **Linear scaling rule** — LR should scale with batch size
8. **Generalization gap** — Small batches generalize better than large batches
9. **Distributed SGD** — Gradient averaging across workers
10. **Critical batch size** — Diminishing returns beyond a certain batch size

See [exercises.ipynb](exercises.ipynb) for graded problems on these topics.